Cell 1: Imports

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Cell 2: Load model

In [ ]:
xgb_model = joblib.load("data/models/xgb_temporal.joblib")
print(type(xgb_model))

Cell 3: Load temporal dataset

In [ ]:
feature_df = pd.read_parquet("data/processed/feature_df.parquet")  # change path if needed
feature_df["day"] = pd.to_datetime(feature_df["day"], errors="coerce")
feature_df = feature_df.sort_values(["day", "user"]).reset_index(drop=True)

print(feature_df.shape)

Cell 4: Set columns

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


TARGET_COL = "is_insider"
GROUP_COL = "user"
TIME_COL = "day"

leak_cols = [
    "after_hours_events",
    "user_mean_after",
    "user_std_after",
    "after_hours_events_dev",
    "after_hours_ratio",
]
mean_std_cols = [c for c in feature_df.columns if c.startswith("user_mean_") or c.startswith("user_std_")]
# drop_cols = list(set(leak_cols + mean_std_cols + ["user", "day", "employee_name", "email", "projects"]))
drop_cols = list(set(leak_cols + mean_std_cols + ["user", "day", "employee_name", "email", "projects", "total_events"]))

feature_cols = [c for c in feature_df.columns if c not in drop_cols]

X = feature_df[feature_cols].copy()
y = feature_df[TARGET_COL].copy()
groups = feature_df[GROUP_COL].copy()

print("Number of features:", len(feature_cols))
print(feature_cols[:20])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(feature_df, feature_df["is_insider"], groups=groups))

Cell 5: Load X_train, X_test, y_train, y_test datasets

In [ ]:
X_train = pd.read_parquet("data/processed/X_train.parquet")
y_train = pd.read_csv("data/processed/y_train.csv").squeeze("columns")
X_test = pd.read_parquet("data/processed/X_test.parquet")
y_test = pd.read_csv("data/processed/y_test.csv").squeeze("columns")

X_train = X_train.drop(columns='total_events')
X_test = X_test.drop(columns='total_events')

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Train target distribution:\n", y_train.value_counts())
print("Test target distribution:\n", y_test.value_counts())

Cell 6: Predict probabilities

In [ ]:
y_score = xgb_model.predict_proba(X_test)[:, 1]

topk_df = X_test.copy()
topk_df[TARGET_COL] = y_test.values
topk_df["pred_proba"] = y_score
topk_df["rank"] = topk_df["pred_proba"].rank(method="first", ascending=False).astype(int)

topk_df = topk_df.sort_values("pred_proba", ascending=False).reset_index(drop=True)
topk_df.head(10)

Cell 7: Top-k metrics function

In [ ]:
def compute_topk_metrics(df_scored, target_col, score_col, k_list):
    df_sorted = df_scored.sort_values(score_col, ascending=False).reset_index(drop=True)
    total_positives = df_sorted[target_col].sum()
    base_rate = df_sorted[target_col].mean()
    
    rows = []
    
    for k in k_list:
        top_k = df_sorted.head(k)
        hits = int(top_k[target_col].sum())
        precision_at_k = hits / k if k > 0 else 0.0
        recall_at_k = hits / total_positives if total_positives > 0 else 0.0
        lift_at_k = precision_at_k / base_rate if base_rate > 0 else np.nan
        
        rows.append({
            "k": k,
            "hits_at_k": hits,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "lift_at_k": lift_at_k
        })
    
    return pd.DataFrame(rows)

Cell 8: Choose K values

In [ ]:
k_list = [5, 10, 20, 50, 100, 200, 500]
topk_results = compute_topk_metrics(topk_df, TARGET_COL, "pred_proba", k_list)
topk_results

Cell 9: Cumulative recall curve

In [ ]:
topk_df["cum_hits"] = topk_df[TARGET_COL].cumsum()
total_positives = topk_df[TARGET_COL].sum()
topk_df["cum_recall"] = topk_df["cum_hits"] / total_positives if total_positives > 0 else 0

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(topk_df) + 1), topk_df["cum_recall"], linewidth=2)
plt.xlabel("Top-K Ranked Records")
plt.ylabel("Cumulative Recall")
plt.title("Top-K Cumulative Recall for XGBoost")
plt.tight_layout()
plt.show()

Cell 10: Precision@K plot

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=topk_results, x="k", y="precision_at_k", color="steelblue")
plt.title("Precision@K for XGBoost")
plt.ylabel("Precision@K")
plt.xlabel("K")
plt.tight_layout()
plt.show()

Cell 11: Recall@K plot

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=topk_results, x="k", y="recall_at_k", marker="o", linewidth=2)
plt.title("Recall@K for XGBoost")
plt.ylabel("Recall@K")
plt.xlabel("K")
plt.tight_layout()
plt.show()

Cell 12: Lift@K plot

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=topk_results, x="k", y="lift_at_k", marker="o", linewidth=2, color="darkgreen")
plt.title("Lift@K for XGBoost")
plt.ylabel("Lift@K")
plt.xlabel("K")
plt.tight_layout()
plt.show()

Cell 13: Inspect top flagged records

In [ ]:
top_records = topk_df.reset_index(drop=True).copy()

cols_to_show = ["pred_proba", TARGET_COL]
display_cols = [c for c in [GROUP_COL, TIME_COL] if c in top_records.columns] + cols_to_show
top_records[display_cols].head(20)

Cell 14: Summary statements

In [ ]:
topk_results_rounded = topk_results.copy()
for col in ["precision_at_k", "recall_at_k", "lift_at_k"]:
    topk_results_rounded[col] = topk_results_rounded[col].round(4)

topk_results_rounded

Cell 15: Save outputs

In [ ]:
topk_results.to_csv("data/processed/topk_results_xgb.csv", index=False)
topk_df.to_csv("data/processed/topk_ranked_predictions_xgb.csv", index=False)

print("Saved top-k results and ranked predictions.")